# Kaggle Test Evaluation

Notebook nay evaluate checkpoint tren tap `test-00000-of-00001.parquet` ngay tren Kaggle.

Default hien tai: doc old output dataset `anhnguyen0812/nlp-vit5-ep3-old-output`, uu tien root `/kaggle/input/nlp-vit5-ep3-old-output/summarization_outputs`, fallback sang `/kaggle/input/datasets/anhnguyen0812/nlp-vit5-ep3-old-output/summarization_outputs`, roi evaluate cac run match `RUN_GLOB='*vit5*'` tren full test.

Neu muon quay lai evaluate output kernel `anhnguyenphi/nlp-aio`, them slug vao `KAGGLE_KERNEL_OUTPUTS`, bat `DOWNLOAD_KERNEL_OUTPUTS=True`, va doi `RUN_GLOB` phu hop.


In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
from zipfile import ZipFile

PROJECT_NAME = 'pretrained-summarization'
REPO_URL = 'https://github.com/Anhnguyen0812/pretrained-summarization.git'
REFRESH_REPO = True
WORKING = Path('/kaggle/working')
WORKING_REPO = WORKING / PROJECT_NAME
RUNS_ROOT = WORKING / 'test_eval_runs'
KERNEL_OUTPUT_DIR = WORKING / 'kernel_outputs'
KAGGLE_KERNEL_OUTPUTS = []  # optional: add 'anhnguyenphi/nlp-aio' neu muon tai kernel output moi
KAGGLE_INPUT_OUTPUT_DATASETS = [
    'nlp-vit5-ep3-old-output',
]
EXPLICIT_OUTPUT_ROOTS = [
    Path('/kaggle/input/nlp-vit5-ep3-old-output/summarization_outputs'),
    Path('/kaggle/input/datasets/anhnguyen0812/nlp-vit5-ep3-old-output/summarization_outputs'),
]
DOWNLOAD_KERNEL_OUTPUTS = False
MAX_DOWNLOAD_ATTEMPTS = 2
PARALLEL_EVAL = True
EVAL_BATCH_SIZE = 16  # seq2seq ViT5/BARTpho full test nhanh; giam ve 2-4 neu test causal LM
OUT_DIR = WORKING / 'test_eval_outputs'
TEST_FILE = Path('/kaggle/input/datasets/anhnguyen0812/nlp-vietnamese-sumarization/test-00000-of-00001.parquet')
TEST_BASENAME = TEST_FILE.name
MAX_TEST_SAMPLES = None  # None = full test; dat 300/500 neu muon smoke test nhanh
RUN_GLOB = '*vit5*'  # mac dinh test cac run ViT5 trong dataset old output; doi '*' de test tat ca
FAST_GENERATION = False  # False = dung generation config cua checkpoint; bat True neu test causal LM nhanh
FAST_MAX_NEW_TOKENS = 96
FAST_NUM_BEAMS = 1

def run(cmd, cwd=None, check=True):
    print('CMD:', cmd, flush=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(cmd, shell=True, cwd=str(cwd) if cwd else None, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in process.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    code = process.wait()
    if check and code != 0:
        raise RuntimeError(f'Command failed with exit code {code}: {cmd}\nLast lines:\n{"".join(lines[-100:])}')
    return code

def is_repo(path):
    return (path / 'pyproject.toml').exists() and (path / 'src' / 'vn_summarization').exists()

WORKING.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
if REFRESH_REPO and WORKING_REPO.exists():
    shutil.rmtree(WORKING_REPO)
if not is_repo(WORKING_REPO):
    run(f'git clone --depth 1 {REPO_URL} {WORKING_REPO}', cwd=WORKING)
else:
    run('git pull --ff-only', cwd=WORKING_REPO, check=False)
repo = WORKING_REPO
run('git log --oneline -1', cwd=repo)


CMD: git clone --depth 1 https://github.com/Anhnguyen0812/pretrained-summarization.git /kaggle/working/pretrained-summarization
Cloning into '/kaggle/working/pretrained-summarization'...
CMD: git log --oneline -1
437b4be Prefer UI-mounted old ViT5 output root


0

In [2]:
os.chdir(repo)
run(f'{sys.executable} -m pip install -q --upgrade pip', cwd=repo)
run(f'{sys.executable} -m pip install -q -e .', cwd=repo)
run(f'{sys.executable} -m pip install -q --upgrade "transformers>=4.51.0,<5" "tokenizers>=0.22.0,<=0.23.0"', cwd=repo)
run(f'{sys.executable} -m pip check', cwd=repo, check=False)
run(f'{sys.executable} -m pip show transformers tokenizers peft accelerate | sed -n "/Name: /p;/Version: /p"', cwd=repo, check=False)
run(f"{sys.executable} -c 'import tokenizers, transformers; print(\"TRANSFORMERS\", transformers.__version__); print(\"TOKENIZERS\", tokenizers.__version__)'", cwd=repo)
run('nvidia-smi', check=False)


CMD: /usr/bin/python3 -m pip install -q --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.2 MB/s eta 0:00:00
CMD: /usr/bin/python3 -m pip install -q -e .
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompa

0

## Prepare Runs And Test File

Notebook mac dinh doc old output dataset `nlp-vit5-ep3-old-output` da attach trong `/kaggle/input`. Neu can, van co the them kernel slug vao `KAGGLE_KERNEL_OUTPUTS`. Moi run co `resolved_config.json` va `best/checkpoint` co model artifact se duoc gom vao `/kaggle/working/test_eval_runs`.


In [3]:
EXPECTED_OUTPUT_ROOT_NAMES = ['summarization_outputs', 'report_experiment_outputs']
RESULT_ZIP_NAMES = ['allinone_report_results.zip', 'summarization_results.zip', 'causal_lm_results.zip']

def find_files(root, pattern):
    root = Path(root)
    if not root.exists():
        return []
    return sorted(root.rglob(pattern))

def download_kernel_outputs():
    if not DOWNLOAD_KERNEL_OUTPUTS:
        return
    KERNEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for kernel in KAGGLE_KERNEL_OUTPUTS:
        dest = KERNEL_OUTPUT_DIR / kernel.split('/')[-1]
        dest.mkdir(parents=True, exist_ok=True)
        done_marker = dest / '.download_complete'
        if done_marker.exists():
            print('KERNEL OUTPUT COMPLETE:', kernel, dest)
            continue
        for attempt in range(1, MAX_DOWNLOAD_ATTEMPTS + 1):
            print(f'DOWNLOAD ATTEMPT {attempt}/{MAX_DOWNLOAD_ATTEMPTS}:', kernel)
            code = run(f'kaggle kernels output {kernel} -p {dest}', cwd=WORKING, check=False)
            if code == 0:
                done_marker.write_text('ok', encoding='utf-8')
                break
            print('WARN: could not fully download kernel output:', kernel)
        if not done_marker.exists():
            print('WARN: using partial output if usable:', dest)

def unzip_result_zips():
    zip_candidates = []
    search_roots = [KERNEL_OUTPUT_DIR, Path('/kaggle/input'), Path('/kaggle/working')]
    for root in search_roots:
        if not root.exists():
            continue
        for name in RESULT_ZIP_NAMES:
            zip_candidates.extend(root.rglob(name))
        zip_candidates.extend(p for p in root.rglob('*.zip') if p.name.startswith(('allinone_', 'summarization_', 'causal_lm_')))
    zip_candidates = sorted(set(p for p in zip_candidates if p.name != 'test_eval_results.zip'))
    if not zip_candidates:
        print('No result zip found. Will try direct kernel output folders.')
        return
    for zip_path in zip_candidates:
        print('UNZIP', zip_path, '->', WORKING)
        with ZipFile(zip_path) as zf:
            zf.extractall(WORKING)

def candidate_output_roots():
    roots = []
    for explicit_root in EXPLICIT_OUTPUT_ROOTS:
        print('EXPLICIT_OUTPUT_ROOT:', explicit_root, explicit_root.exists())
        if explicit_root.exists():
            roots.append(explicit_root)
    search_roots = [KERNEL_OUTPUT_DIR, WORKING]
    input_root = Path('/kaggle/input')
    if input_root.exists():
        if KAGGLE_INPUT_OUTPUT_DATASETS:
            for dataset_name in KAGGLE_INPUT_OUTPUT_DATASETS:
                dataset_root = input_root / dataset_name
                print('INPUT_OUTPUT_DATASET:', dataset_root, dataset_root.exists())
                if dataset_root.exists():
                    search_roots.append(dataset_root)
        else:
            search_roots.append(input_root)
    for base in search_roots:
        if not base.exists():
            continue
        for root_name in EXPECTED_OUTPUT_ROOT_NAMES:
            direct = base / root_name
            if direct.exists():
                roots.append(direct)
            roots.extend(p for p in base.rglob(root_name) if p.is_dir())
    unique = []
    seen = set()
    for root in sorted(roots):
        key = str(root.resolve())
        if key not in seen:
            seen.add(key)
            unique.append(root)
    return unique

def has_model_artifacts(model_dir):
    return any((model_dir / name).exists() for name in ['adapter_model.safetensors', 'model.safetensors', 'pytorch_model.bin'])

def model_dir_for_run(run_dir):
    best = run_dir / 'best'
    if best.exists() and has_model_artifacts(best):
        return best
    checkpoints = sorted(
        (p for p in run_dir.glob('checkpoint-*') if p.is_dir()),
        key=lambda p: int(p.name.split('-')[-1]) if p.name.split('-')[-1].isdigit() else -1,
        reverse=True,
    )
    for checkpoint in checkpoints:
        if has_model_artifacts(checkpoint):
            return checkpoint
    return None

def collect_run_dirs(output_roots):
    found = []
    for root in output_roots:
        for run_dir in sorted(p for p in root.iterdir() if p.is_dir()):
            if (run_dir / 'resolved_config.json').exists() and model_dir_for_run(run_dir) is not None:
                found.append(run_dir)
    unique = []
    seen = set()
    for run_dir in sorted(found):
        key = str(run_dir.resolve())
        if key not in seen:
            seen.add(key)
            unique.append(run_dir)
    return unique

def build_merged_runs_root(run_dirs):
    if RUNS_ROOT.exists():
        shutil.rmtree(RUNS_ROOT)
    RUNS_ROOT.mkdir(parents=True, exist_ok=True)
    for idx, run_dir in enumerate(run_dirs):
        name = run_dir.name
        dest = RUNS_ROOT / name
        if dest.exists():
            dest = RUNS_ROOT / f'{name}_{idx}'
        try:
            os.symlink(run_dir, dest, target_is_directory=True)
        except OSError:
            shutil.copytree(run_dir, dest)
        print('ADD RUN:', dest.name, '<-', run_dir)

def find_test_file():
    if TEST_FILE.exists():
        return TEST_FILE
    candidates = find_files('/kaggle/input', TEST_BASENAME) + find_files('/kaggle/working', TEST_BASENAME)
    if not candidates:
        raise FileNotFoundError(f'Khong thay {TEST_FILE} hoac file ten {TEST_BASENAME}. Hay attach/upload Kaggle dataset co test parquet.')
    return candidates[0]

download_kernel_outputs()
unzip_result_zips()
output_roots = candidate_output_roots()
print('OUTPUT_ROOT_CANDIDATES:', output_roots)
run_dirs = collect_run_dirs(output_roots)
if not run_dirs:
    raise FileNotFoundError('Khong thay run folder nao co resolved_config.json va best/checkpoint co *.safetensors trong summarization_outputs/report_experiment_outputs. Kernel output co the bi tai dut; hay rerun cell download hoac attach notebook output/dataset day du.')
build_merged_runs_root(run_dirs)
TEST_FILE = find_test_file()
merged_run_dirs = sorted(p for p in RUNS_ROOT.iterdir() if p.is_dir() and (p / 'resolved_config.json').exists())
print('TEST_FILE:', TEST_FILE)
print('RUNS_ROOT:', RUNS_ROOT)
print('RUNS:', [p.name for p in merged_run_dirs])


No result zip found. Will try direct kernel output folders.
EXPLICIT_OUTPUT_ROOT: /kaggle/input/nlp-vit5-ep3-old-output/summarization_outputs False
EXPLICIT_OUTPUT_ROOT: /kaggle/input/datasets/anhnguyen0812/nlp-vit5-ep3-old-output/summarization_outputs True
INPUT_OUTPUT_DATASET: /kaggle/input/nlp-vit5-ep3-old-output False
OUTPUT_ROOT_CANDIDATES: [PosixPath('/kaggle/input/datasets/anhnguyen0812/nlp-vit5-ep3-old-output/summarization_outputs')]
ADD RUN: vit5_base_ep3_t4x2 <- /kaggle/input/datasets/anhnguyen0812/nlp-vit5-ep3-old-output/summarization_outputs/vit5_base_ep3_t4x2
ADD RUN: vit5_base_lora_ep3_t4x2 <- /kaggle/input/datasets/anhnguyen0812/nlp-vit5-ep3-old-output/summarization_outputs/vit5_base_lora_ep3_t4x2
TEST_FILE: /kaggle/input/datasets/anhnguyen0812/nlp-vietnamese-sumarization/test-00000-of-00001.parquet
RUNS_ROOT: /kaggle/working/test_eval_runs
RUNS: ['vit5_base_ep3_t4x2', 'vit5_base_lora_ep3_t4x2']


## Evaluate On Test

Mac dinh notebook chay old ViT5 input dataset: `RUN_GLOB="*vit5*"`, `MAX_TEST_SAMPLES=None`, `EVAL_BATCH_SIZE=16`, va khong override generation config. Day la che do full test cho ViT5/BARTpho. Neu test causal LM, giam batch ve 2-4 va bat `FAST_GENERATION=True`.


In [4]:
import csv
import json
import torch

SHARD_ROOT = WORKING / 'test_eval_shards'
if SHARD_ROOT.exists():
    shutil.rmtree(SHARD_ROOT)
SHARD_ROOT.mkdir(parents=True, exist_ok=True)

all_runs = sorted(p for p in RUNS_ROOT.glob(RUN_GLOB) if p.is_dir() and (p / 'resolved_config.json').exists())
if not all_runs:
    raise FileNotFoundError(f'No runs under {RUNS_ROOT}')
print('RUN_GLOB:', RUN_GLOB)
print('MAX_TEST_SAMPLES:', MAX_TEST_SAMPLES)
print('SELECTED_RUNS:', [p.name for p in all_runs])

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
gpus = list(range(gpu_count)) if (PARALLEL_EVAL and gpu_count >= 2) else [0 if gpu_count == 1 else None]
print('GPU_COUNT:', gpu_count, 'EVAL_GPUS:', gpus)

shard_roots = []
for shard_idx, gpu in enumerate(gpus):
    shard_root = SHARD_ROOT / f'shard_{shard_idx}'
    shard_root.mkdir(parents=True, exist_ok=True)
    shard_roots.append((shard_root, gpu))

for idx, run_dir in enumerate(all_runs):
    shard_root, _gpu = shard_roots[idx % len(shard_roots)]
    dest = shard_root / run_dir.name
    try:
        os.symlink(run_dir, dest, target_is_directory=True)
    except OSError:
        shutil.copytree(run_dir, dest)
    print('SHARD ADD:', shard_root.name, run_dir.name)

processes = []
for shard_idx, (shard_root, gpu) in enumerate(shard_roots):
    if not any(shard_root.iterdir()):
        continue
    shard_out = OUT_DIR / f'shard_{shard_idx}'
    args = f'--runs_root {shard_root} --test_file {TEST_FILE} --out_dir {shard_out} --eval_batch_size {EVAL_BATCH_SIZE}'
    if FAST_GENERATION:
        args += f' --generation_max_new_tokens {FAST_MAX_NEW_TOKENS} --generation_num_beams {FAST_NUM_BEAMS}'
    if MAX_TEST_SAMPLES:
        args += f' --max_test_samples {MAX_TEST_SAMPLES}'
    cmd = f'{sys.executable} -u -m vn_summarization.evaluate_runs_on_test {args}'
    env = os.environ.copy()
    if gpu is not None:
        env['CUDA_VISIBLE_DEVICES'] = str(gpu)
    print('LAUNCH:', cmd, 'CUDA_VISIBLE_DEVICES=', env.get('CUDA_VISIBLE_DEVICES'))
    processes.append((shard_idx, subprocess.Popen(cmd, shell=True, cwd=str(repo), env=env)))

for shard_idx, process in processes:
    code = process.wait()
    if code != 0:
        raise RuntimeError(f'Eval shard {shard_idx} failed with exit code {code}')


def to_float(value):
    try:
        return float(value)
    except Exception:
        return float('-inf')

rows = []
for csv_path in sorted(OUT_DIR.glob('shard_*/test_results.csv')):
    with csv_path.open('r', encoding='utf-8', newline='') as f:
        rows.extend(csv.DictReader(f))
rows.sort(key=lambda row: to_float(row.get('rougeL')), reverse=True)
if not rows:
    raise RuntimeError('No shard test results found')

columns = list(rows[0].keys())
merged_csv = OUT_DIR / 'test_results.csv'
with merged_csv.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()
    for row in rows:
        writer.writerow(row)

md_columns = ['run', 'kind', 'model', 'rouge1', 'rouge2', 'rougeL', 'loss', 'gen_len']
lines = ['# Test Results', '', '| ' + ' | '.join(md_columns) + ' |', '| ' + ' | '.join(['---'] * len(md_columns)) + ' |']
for row in rows:
    lines.append('| ' + ' | '.join(str(row.get(col, '')) for col in md_columns) + ' |')
lines.append('')
merged_md = OUT_DIR / 'test_results.md'
merged_md.write_text('\n'.join(lines), encoding='utf-8')
(OUT_DIR / 'best_test_run.json').write_text(json.dumps(rows[0], ensure_ascii=False, indent=2), encoding='utf-8')
print(merged_md.read_text(encoding='utf-8'))
print('MERGED CSV:', merged_csv)
print('BEST:', OUT_DIR / 'best_test_run.json')


RUN_GLOB: *vit5*
MAX_TEST_SAMPLES: None
SELECTED_RUNS: ['vit5_base_ep3_t4x2', 'vit5_base_lora_ep3_t4x2']
GPU_COUNT: 2 EVAL_GPUS: [0, 1]
SHARD ADD: shard_0 vit5_base_ep3_t4x2
SHARD ADD: shard_1 vit5_base_lora_ep3_t4x2
LAUNCH: /usr/bin/python3 -u -m vn_summarization.evaluate_runs_on_test --runs_root /kaggle/working/test_eval_shards/shard_0 --test_file /kaggle/input/datasets/anhnguyen0812/nlp-vietnamese-sumarization/test-00000-of-00001.parquet --out_dir /kaggle/working/test_eval_outputs/shard_0 --eval_batch_size 16 CUDA_VISIBLE_DEVICES= 0
LAUNCH: /usr/bin/python3 -u -m vn_summarization.evaluate_runs_on_test --runs_root /kaggle/working/test_eval_shards/shard_1 --test_file /kaggle/input/datasets/anhnguyen0812/nlp-vietnamese-sumarization/test-00000-of-00001.parquet --out_dir /kaggle/working/test_eval_outputs/shard_1 --eval_batch_size 16 CUDA_VISIBLE_DEVICES= 1


2026-06-10 18:44:04.862771: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-10 18:44:04.862863: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781117045.274978      98 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781117045.275073      99 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781117045.375471      99 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
E0000 00:00:1781117045.375492      98 cuda_blas.cc:1

# Test Results

| run | kind | model | rouge1 | rouge2 | rougeL | loss | gen_len |
| --- | --- | --- | --- | --- | --- | --- | --- |
| vit5_base_lora_ep3_t4x2 | seq2seq | VietAI/vit5-base | 72.6211 | 44.083 | 46.6347 | 1.1023026704788208 | 148.0476 |

CSV: /kaggle/working/test_eval_outputs/shard_1/test_results.csv
Best: /kaggle/working/test_eval_outputs/shard_1/best_test_run.json
# Test Results

| run | kind | model | rouge1 | rouge2 | rougeL | loss | gen_len |
| --- | --- | --- | --- | --- | --- | --- | --- |
| vit5_base_ep3_t4x2 | seq2seq | VietAI/vit5-base | 74.2161 | 46.7484 | 48.8942 | 0.972809910774231 | 154.381 |

CSV: /kaggle/working/test_eval_outputs/shard_0/test_results.csv
Best: /kaggle/working/test_eval_outputs/shard_0/best_test_run.json
# Test Results

| run | kind | model | rouge1 | rouge2 | rougeL | loss | gen_len |
| --- | --- | --- | --- | --- | --- | --- | --- |
| vit5_base_ep3_t4x2 | seq2seq | VietAI/vit5-base | 74.2161 | 46.7484 | 48.8942 | 0.972809910774231 | 154.3

In [5]:
zip_path = WORKING / 'test_eval_results.zip'
if zip_path.exists():
    zip_path.unlink()
keep_suffixes = {'.json', '.jsonl', '.csv', '.md', '.txt'}
files = [p for p in OUT_DIR.rglob('*') if p.is_file() and p.suffix in keep_suffixes]
with ZipFile(zip_path, 'w') as zf:
    for file in files:
        zf.write(file, file.relative_to(WORKING).as_posix())
print('ZIP', zip_path)
print('TEST CSV', OUT_DIR / 'test_results.csv')
print('TEST MD', OUT_DIR / 'test_results.md')
for file in sorted(files):
    print(file)


ZIP /kaggle/working/test_eval_results.zip
TEST CSV /kaggle/working/test_eval_outputs/test_results.csv
TEST MD /kaggle/working/test_eval_outputs/test_results.md
/kaggle/working/test_eval_outputs/best_test_run.json
/kaggle/working/test_eval_outputs/shard_0/best_test_run.json
/kaggle/working/test_eval_outputs/shard_0/test_results.csv
/kaggle/working/test_eval_outputs/shard_0/test_results.md
/kaggle/working/test_eval_outputs/shard_0/vit5_base_ep3_t4x2/predictions_test.jsonl
/kaggle/working/test_eval_outputs/shard_0/vit5_base_ep3_t4x2/resolved_test_config.json
/kaggle/working/test_eval_outputs/shard_0/vit5_base_ep3_t4x2/test_metrics.json
/kaggle/working/test_eval_outputs/shard_0/vit5_base_ep3_t4x2/validation_metrics.json
/kaggle/working/test_eval_outputs/shard_1/best_test_run.json
/kaggle/working/test_eval_outputs/shard_1/test_results.csv
/kaggle/working/test_eval_outputs/shard_1/test_results.md
/kaggle/working/test_eval_outputs/shard_1/vit5_base_lora_ep3_t4x2/predictions_test.jsonl
/kaggle